# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jericho-Ram/FlyRank-Internship-ML/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The queue ranks the same population from Weeks 5-6 (26,604 rows, base rate 0.611) using two
independent signals side by side, instead of collapsing them into one black-box number: the
Week-4 hand rule (missed clicks against tier-expected CTR) and the Week-5 RandomForest,
re-scored here out-of-fold on the exact grouped folds from Week 6 (permissive OOF AUC 0.662,
strict OOF AUC 0.628 -- pooled across all folds, not the per-fold mean Week 6 quotes, so the
numbers move slightly but tell the same story).

Four tiers, in priority order:

1. **Refresh now** (5,396 rows) -- rule and model agree the page is underperforming. Sorted
   within tier by the rule's missed-click share, since that number is tied to an actual
   measured traffic gap, not just a probability.
2. **Investigate -- rule silent, model flags it** (11,276 rows) -- the rule sees no CTR gap
   (usually because expected clicks are too small to score, or there genuinely is no gap), but
   the model puts p(decline) >= 0.6. This is the subset Week 6 measured the model adding real
   value on top of the rule; here it reads a 70.2% true-decline rate, close to Week 6's 60.1%
   figure on the full rule-silent population -- this narrower p >= 0.6 cut is more selective,
   which is why it reads higher. Flagged for investigation, not automatic action -- see
   Section 3.
3. **CTR gap only, model unsure** (1,730 rows) -- the rule found a real, measured click gap
   against tier-expected CTR even though the model's decline probability is under 0.5. Kept in
   the queue because the rule answers a different question (are we getting the clicks this
   position should get) than the model (is this page trending down) -- a page can lag its
   position's expected CTR while its 30-day trend still looks stable.
4. **Monitor** (8,202 rows) -- neither signal flags a problem right now.

Reason codes (multi-label, attached to every row in the exported queue): `rule_and_model_agree`,
`model_flags_rule_silent`, `rule_flags_model_unsure`, `high_confidence_model` (p >= 0.7),
`window_reliant_caveat` (Section 2), `top_visible_tier`, `outside_rule_scope` (Section 2), and
`recently_updated_flag` (Section 3) -- a human scanning the queue reads the codes, not the model
internals, to know why a row landed where it did.

In [1]:
import os
import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
np.random.seed(SEED)

if os.getcwd().replace("\\", "/").endswith("work/notebooks"):
    os.chdir("../..")

CSV_PATH = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(CSV_PATH), "starter CSV not found -- are you at the repo root?"
df = pd.read_csv(CSV_PATH)

# --- Rebuild the Week-5/6 population, label, feature sets ------------------
ranked = df["avg_position"] > 0
can_decline = df["impressions_prev_30d"] > 0
pop = df[ranked & can_decline].reset_index(drop=True).copy()
y = pop["trend_direction"].str.lower().eq("down").to_numpy()
groups = pop["client_id"].to_numpy()

LEAK_COLS = ["trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d"]
WINDOW_COLS = ["clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d"]
DROP_ALWAYS = ["content_id", "client_id"] + LEAK_COLS
FEATURES = [c for c in pop.columns if c not in DROP_ALWAYS]
STRICT_FEATURES = [c for c in FEATURES if c not in WINDOW_COLS]

print(f"population: {len(pop):,} rows | permissive features: {len(FEATURES)} | base rate: {y.mean():.3f}")

# --- Week-4 rule, re-scored on this population (same as w05) ---------------
tier_stats = df[ranked].groupby("position_tier").agg(ti=("impressions_90d", "sum"), tc=("clicks_90d", "sum"))
TIER_CTR = tier_stats["tc"] / tier_stats["ti"]
EXPECTED_CLICKS_FLOOR = 5.0
IN_SCOPE_TIERS = ["page_1", "striking", "page_3_5", "top_3"]
pop["tier_expected_ctr"] = pop["position_tier"].map(TIER_CTR)
pop["expected_clicks"] = pop["impressions_90d"] * pop["tier_expected_ctr"]
pop["missed_clicks"] = pop["expected_clicks"] - pop["clicks_90d"]
_in_scope = pop["position_tier"].isin(IN_SCOPE_TIERS)
_scorable = _in_scope & (pop["expected_clicks"] >= EXPECTED_CLICKS_FLOOR)
_fires = _scorable & (pop["missed_clicks"] > 0)
pop["baseline_score"] = np.where(_fires, pop["missed_clicks"] / pop["expected_clicks"], 0.0)
pop["rule_in_scope"] = _in_scope


def build_pipe(model, cols):
    num = [c for c in cols if pd.api.types.is_numeric_dtype(pop[c])]
    cat = [c for c in cols if c not in num]
    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), cat),
    ])
    return Pipeline([("pre", pre), ("model", model)])


GKF = GroupKFold(n_splits=5)
folds = list(GKF.split(pop, y, groups))


def cv_oof(cols):
    oof = np.zeros(len(pop))
    for tr, te in folds:
        rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=SEED)
        pipe = build_pipe(rf, cols)
        pipe.fit(pop[cols].iloc[tr], y[tr])
        oof[te] = pipe.predict_proba(pop[cols].iloc[te])[:, 1]
    return oof


# --- Honest out-of-fold model scores (same model as w05/w06) ---------------
p_permissive = cv_oof(FEATURES)
p_strict = cv_oof(STRICT_FEATURES)
print(f"\nout-of-fold ROC AUC, permissive : {roc_auc_score(y, p_permissive):.3f}")
print(f"out-of-fold ROC AUC, strict     : {roc_auc_score(y, p_strict):.3f}")
print("(pooled OOF AUC, not the per-fold mean quoted in w06 -- same model, same folds, different aggregation)")

# --- Build the ranked queue: rule x model agreement, plus reason codes -----
q = pop[["content_id", "client_id", "position_tier", "rule_in_scope", "baseline_score",
         "impressions_90d", "clicks_90d", "avg_position", "content_age_days",
         "days_since_last_update", "content_type"]].copy()
q["truth_declining"] = y
q["p_decline"] = p_permissive
q["p_decline_strict"] = p_strict
q["window_reliance"] = q["p_decline"] - q["p_decline_strict"]

VISIBLE_TIERS = ["page_1", "top_3", "striking"]
q["rule_flags"] = q["baseline_score"] > 0.0
q["model_flags"] = q["p_decline"] >= 0.5
q["window_reliant"] = q["window_reliance"] >= 0.05
q["top_visible"] = q["position_tier"].isin(VISIBLE_TIERS)
q["high_confidence_model"] = q["p_decline"] >= 0.7
q["recently_updated"] = q["days_since_last_update"] < 14


def action_tier(row):
    if row["rule_flags"] and row["model_flags"]:
        return "1_refresh_now"
    if (not row["rule_flags"]) and row["p_decline"] >= 0.6:
        return "2_investigate_rule_silent"
    if row["rule_flags"] and not row["model_flags"]:
        return "3_ctr_gap_only"
    return "4_monitor"


q["action_tier"] = q.apply(action_tier, axis=1)


def reason_codes(row):
    codes = []
    if row["rule_flags"] and row["model_flags"]:
        codes.append("rule_and_model_agree")
    if (not row["rule_flags"]) and row["p_decline"] >= 0.6:
        codes.append("model_flags_rule_silent")
    if row["rule_flags"] and not row["model_flags"]:
        codes.append("rule_flags_model_unsure")
    if row["high_confidence_model"]:
        codes.append("high_confidence_model")
    if row["window_reliant"]:
        codes.append("window_reliant_caveat")
    if row["top_visible"]:
        codes.append("top_visible_tier")
    if not row["rule_in_scope"]:
        codes.append("outside_rule_scope")
    if row["recently_updated"]:
        codes.append("recently_updated_flag")
    return "|".join(codes)


q["reason_codes"] = q.apply(reason_codes, axis=1)
q_sorted = q.sort_values(["action_tier", "baseline_score"], ascending=[True, False]).reset_index(drop=True)
q_sorted.insert(0, "rank", np.arange(1, len(q_sorted) + 1))

print("\n--- action tier counts ---")
print(q_sorted["action_tier"].value_counts().sort_index().to_string())

print("\n--- top 5 rows ---")
print(q_sorted[["rank", "content_id", "action_tier", "baseline_score", "p_decline", "reason_codes"]]
      .head(5).to_string(index=False))

population: 26,604 rows | permissive features: 38 | base rate: 0.611



out-of-fold ROC AUC, permissive : 0.662
out-of-fold ROC AUC, strict     : 0.628
(pooled OOF AUC, not the per-fold mean quoted in w06 -- same model, same folds, different aggregation)



--- action tier counts ---
action_tier
1_refresh_now                 5396
2_investigate_rule_silent    11276
3_ctr_gap_only                1730
4_monitor                     8202

--- top 5 rows ---
 rank           content_id   action_tier  baseline_score  p_decline                                                                      reason_codes
    1 content_9983d31c53cb 1_refresh_now             1.0   0.735657 rule_and_model_agree|high_confidence_model|top_visible_tier|recently_updated_flag
    2 content_e9785b5bd320 1_refresh_now             1.0   0.663164                                             rule_and_model_agree|top_visible_tier
    3 content_bb2273ff6eed 1_refresh_now             1.0   0.774652                       rule_and_model_agree|high_confidence_model|top_visible_tier
    4 content_c89e3b5466ba 1_refresh_now             1.0   0.771320                       rule_and_model_agree|high_confidence_model|top_visible_tier
    5 content_aa2440f8c4ea 1_refresh_now          

## 2. Intended use and limits

*Who uses this, for what -- and where it stops being valid.*

**Who:** a content/SEO editor deciding what to open first this sprint, across the clients in
this dataset. It's a triage queue, not an auto-publish pipeline -- see Section 3.

**Where it stops being valid:**

- **New clients.** This population covers 31 clients; the smallest has 2 rows, the largest
  is 26.2% of the population. Week 6 measured a 0.122 AUC gap between grouped and random
  splits on this exact model -- meaning a real share of its skill is client-specific pattern
  memorization. For a brand-new client not in this training population, only the
  non-memorized share of that skill should be assumed to carry over; the honest floor is
  closer to the strict OOF AUC than the permissive one.
- **Rows outside the rule's scope.** A block of rows sits in the `deep` position tier, which
  the Week-4 rule never evaluates (it only scores `page_1`, `top_3`, `striking`, `page_3_5`).
  Some of those still got a model-driven action-tier placement (`outside_rule_scope` in reason
  codes) -- read those as model-only opinions with no rule cross-check, not as agreed findings.
- **Window-reliant rows.** A meaningful share of the "refresh now" tier carries the
  `window_reliant_caveat` -- Week 6's Test E found 0.034 AUC is carried by columns
  (`clicks_last_30d`, `sessions_last_30d`, etc.) that share a time window with the label. Their
  rule score is unaffected (it doesn't use those columns), but their model score is partly
  riding on same-window correlation rather than a leading indicator.
- **Snapshot, not forecast.** Both scores describe the current 90/30-day windows in this
  extract. Neither claims a page will keep declining, only that it currently looks like the
  pages that historically did (model) or is currently under-clicking its position (rule).

In [2]:
n_clients = pop["client_id"].nunique()
byc = pop.groupby("client_id").size().sort_values()
print(f"clients in this population : {n_clients}")
print(f"smallest client, rows      : {byc.iloc[0]} ({byc.index[0]})")
print(f"largest client, rows       : {byc.iloc[-1]} ({byc.index[-1]}), {byc.iloc[-1] / len(pop) * 100:.1f}% of population")
print(f"rows outside rule's scope ('deep' tier, rule never evaluates) : {(~pop['rule_in_scope']).sum():,}")

t2 = q_sorted[q_sorted["action_tier"] == "2_investigate_rule_silent"]
print(f"tier 2 (rule silent, model flags) n={len(t2):,}, true declining rate={t2['truth_declining'].mean():.3f}")

t1 = q_sorted[q_sorted["action_tier"] == "1_refresh_now"]
print(f"tier 1 window-reliant share : {t1['window_reliant'].mean():.3f} ({t1['window_reliant'].sum():,} of {len(t1):,})")

clients in this population : 31
smallest client, rows      : 2 (client_1a6562590e)
largest client, rows       : 6981 (client_19581e27de), 26.2% of population
rows outside rule's scope ('deep' tier, rule never evaluates) : 1,136
tier 2 (rule silent, model flags) n=11,276, true declining rate=0.702
tier 1 window-reliant share : 0.229 (1,233 of 5,396)


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on tier 1 or 2:**
- Confirm the page still exists and isn't already superseded or redirected -- the dataset has
  no "deleted" flag.
- Check `recently_updated_flag`: a real slice of tier-1 rows were updated fewer than 14 days
  ago. A page just touched and already flagged again is either a false signal from
  update-day measurement noise, or a real sign the last update didn't work -- a human needs to
  tell which; the model can't.
- Check `outside_rule_scope`: some rows get a tier from the model alone, with no rule
  cross-check (Section 2). Read these as leads, not confirmed findings.
- For any client with thin representation in this population, treat tier assignments as
  low-confidence -- there isn't enough of that client's data in these folds for the grouped-CV
  numbers to say much about them specifically.

**No-go -- never automate:**
- Never auto-publish or auto-edit content from this queue. The output is a prioritized list
  for a human to open, not a content-generation trigger.
- Never treat `high_confidence_model` alone (p >= 0.7, no rule agreement) as sufficient to act
  without a human looking at the actual page -- high confidence is a property of the model's
  probability estimate, not a guarantee of being right, and Section 2's memorization gap means
  some of that confidence is client identity, not content signal.
- Never rank across clients as if scores were comparable in an absolute sense -- the model was
  never tested on a held-out *new* client, only on held-out rows from clients it partly learned
  from via the other folds.

In [3]:
recently_updated_in_t1 = t1["recently_updated"].sum()
print(f"tier 1 rows updated <14 days ago (recently_updated_flag) : {recently_updated_in_t1:,} of {len(t1):,}")

thin_clients = byc[byc < 30]
thin_rows = q_sorted[q_sorted["client_id"].isin(thin_clients.index)]
print(f"clients with <30 rows in this population : {len(thin_clients)} clients, {len(thin_rows):,} rows total")

outside_scope_flagged = q_sorted[
    q_sorted["reason_codes"].str.contains("outside_rule_scope") & (q_sorted["action_tier"] != "4_monitor")
]
print(f"outside-rule-scope rows the model still flagged for action : {len(outside_scope_flagged):,}")

tier 1 rows updated <14 days ago (recently_updated_flag) : 307 of 5,396
clients with <30 rows in this population : 4 clients, 82 rows total
outside-rule-scope rows the model still flagged for action : 356


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Snapshot values from this run, to compare future runs against:

- **Population base rate.** If a future run's base rate moves outside roughly +/-0.05 of this
  run's without a known cause (e.g. a new client added), that's a population-drift trigger --
  the tier thresholds (p >= 0.5, p >= 0.6, p >= 0.7) were picked against this base rate and may
  not mean the same thing against a different one.
- **Tier mix.** A large shift -- especially tier 2 growing much past its current share -- would
  mean the rule is going silent on more of the population than it did here, which is worth
  checking on its own rather than assuming the model is just finding more.
- **Tier-1 window-reliant share.** If this climbs, more of the "refresh now" tier is leaning on
  same-window columns (Section 2) rather than leading indicators -- re-run Week 6's Test E to
  confirm the AUC cost hasn't grown.
- **Retrain trigger.** Re-run the Week-6 grouped-vs-random audit whenever a new client is added
  or the feature set changes. If the gap moves materially past 0.122, that's a sign the model
  is leaning harder on client identity than it was here, and the "new client" caveat in
  Section 2 gets stronger, not weaker.
- **Outcome check (the only trigger that actually validates the queue).** For a sample of
  tier-1 and tier-2 pages a human refreshes, track whether their next 30/90-day trend actually
  improves. This queue has never been checked against a real outcome -- everything above is a
  measured association in one snapshot, not evidence the recommended actions work.

In [4]:
print(f"current population base rate (declining) : {y.mean():.3f}")
tier_mix = q_sorted["action_tier"].value_counts(normalize=True).sort_index()
print("current tier mix:")
print(tier_mix.round(3).to_string())
print(f"current tier-1 window-reliant share          : {t1['window_reliant'].mean():.3f}")

current population base rate (declining) : 0.611
current tier mix:
action_tier
1_refresh_now                0.203
2_investigate_rule_silent    0.424
3_ctr_gap_only               0.065
4_monitor                    0.308
current tier-1 window-reliant share          : 0.229


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [5]:
os.makedirs("work/outputs", exist_ok=True)
export_cols = ["rank", "content_id", "client_id", "action_tier", "reason_codes",
               "baseline_score", "p_decline", "p_decline_strict", "position_tier",
               "impressions_90d", "clicks_90d", "avg_position", "content_age_days",
               "days_since_last_update", "truth_declining"]
q_sorted[export_cols].to_csv("work/outputs/w07_action_queue.csv", index=False)
print(f"wrote work/outputs/w07_action_queue.csv -- {len(q_sorted):,} rows, ranked 1 to {q_sorted['rank'].max():,}")

wrote work/outputs/w07_action_queue.csv -- 26,604 rows, ranked 1 to 26,604


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.